In [ ]:
'''
Sample player names to test:
Cade Cunningham
Lebron James
Stephen Curry
Kevin Durant
Josh Giddey
Nikola Jokic
Anthony Edwards
Derrick White
Payton Pritchard
Cooper Flagg
Jaylen Wells
Jalen Brunson
Pascal Siakam
Jamal Murray
Tyler Herro

For opposing team,
only enter the
city. Dictionary
below has all teams
with corresponding
city
'''

current_season = 2026
player_name = input("Enter Player Name: ")
print("\nFor opponent, enter city only.\nA list of all teams for reference is below.\n")
current_opponent = input("Enter today's opponent: ")

Enter Player Name: Tyler Herro

For opponent, enter city only.
A list of all teams for reference is below.

Enter today's opponent: Washington


In [ ]:
import pandas as pd
import numpy as np
import requests

In [ ]:
ABBR_TO_TEAM = {
    "ATL": "Atlanta",
    "BOS": "Boston",
    "BRK": "Brooklyn",
    "CHO": "Charlotte",
    "CHI": "Chicago",
    "CLE": "Cleveland",
    "DAL": "Dallas",
    "DEN": "Denver",
    "DET": "Detroit",
    "GSW": "Golden State",
    "HOU": "Houston",
    "IND": "Indiana",
    "LAC": "LA Clippers",
    "LAL": "LA Lakers",
    "MEM": "Memphis",
    "MIA": "Miami",
    "MIL": "Milwaukee",
    "MIN": "Minnesota",
    "NOP": "New Orleans",
    "NYK": "New York",
    "OKC": "Oklahoma City",
    "ORL": "Orlando",
    "PHI": "Philadelphia",
    "PHO": "Phoenix",
    "POR": "Portland",
    "SAC": "Sacramento",
    "SAS": "San Antonio",
    "TOR": "Toronto",
    "UTA": "Utah",
    "WAS": "Washington"
}


FEATURE_COLUMNS = [
    # player's offensive stats
    "MP", "FG", "FGA",
    "FG%", "3P", "3PA", "3P%",
    "2P", "2PA", "2P%",
    "eFG%",
    "FT", "FTA", "FT%",
    "ORB", "DRB", "TRB",
    "AST", "TOV"
]

DEF_FEATURE_COLUMNS = [
    # opposing team's defensive stats
    "oPPG",
    "dEFF",
    "pDIFF",
    "eDIFF",
    "PACE",
    "eWIN%"
]

In [ ]:
def make_player_id(name: str):
  # splitting player name into url-friendly split for our data site
    first, last = name.lower().split()
    return f"{last[:5]}{first[:2]}01", last[0]

In [ ]:
def make_weights(n, min_w=0.5, max_w=1.5):
  # weighting recent games heavier than early ones
    return np.linspace(min_w, max_w, n)

In [ ]:
def scrape_bref_gamelog(name: str, year: int):
  # get the url, and get the all games from given season.
  # there are breaks with the value 'Rk' on the site,
  # so this skips that and combines all games into one table
  # returns the final dataset of the season
    player_id, letter = make_player_id(name)
    url = f"https://www.basketball-reference.com/players/{letter}/{player_id}/gamelog/{year}"
    print(url)
    try:
        tables = pd.read_html(url)

        game_log_dfs = []
        for table in tables:
            if 'Rk' in table.columns and 'Date' in table.columns and 'PTS' in table.columns:
                df_clean = table[table['Rk'] != 'Rk'].copy()
                df_clean = df_clean[df_clean['Rk'].notna()].copy()
                if len(df_clean) > 0:
                    df_clean["Season"] = year
                    game_log_dfs.append(df_clean)
        if game_log_dfs:
            df = pd.concat(game_log_dfs, ignore_index=True)
            df = df.iloc[::-1].reset_index(drop=True)
            return df
    except Exception as e:
        print(f"Failed with pandas.read_html: {e}")
        print("You can try a Playwright-based scraper if blocked (requires extra setup).")
    return None

In [ ]:
def get_last_three_seasons(name: str, current_season: int):
  # gets player stats from the last three seasons
    dfs = {}
    for season in [current_season, current_season - 1, current_season - 2]:
        df = scrape_bref_gamelog(name, season)
        if df is not None:
            dfs[season] = df
    if not dfs:
        return None
    return dfs

In [ ]:
def scrape_team_defense(year):
  # gets the table of all team defense for given year
    url = f"https://www.nbastuffer.com/{year-1}-{year}-nba-team-stats/"

    try:
        tables = pd.read_html(url)
        if tables:
            return tables[1]
    except Exception as e:
        print(f"Failed with pandas.read_html: {e}")
        print("You can try a Playwright-based scraper if blocked (requires extra setup).")
    return None

In [ ]:
def get_last_three_def_seasons(current_season):
  # gets team defense stats from the last three seasons
    dfs = {}
    for season in [current_season, current_season - 1, current_season - 2]:
        df = scrape_team_defense(season)
        if df is not None:
            dfs[season] = df
    if not dfs:
        return None
    return dfs

In [ ]:
def min_percentage(mp):
  # turning minutes format into minute percent
    if ":" in mp:
        m, s = mp.split(":")
        return float(m) + float(s) / 60
    return None

In [ ]:
df_def = get_last_three_def_seasons(current_season)
df_off = get_last_three_seasons(player_name, current_season)
merged_seasons = []
y = []

# for cases where nan is present in the table
PCT_COLS = ["FG%", "2P%", "3P%", "FT%", "eFG%"]

for season in df_off.keys():
  # combining defense and offense into one df
  # by matching who they played to that team's defense
    off_df = df_off[season].copy()
    off_df["Opp"] = off_df["Opp"].map(ABBR_TO_TEAM)
    off_df["MP"] = off_df["MP"].map(min_percentage)

    for c in PCT_COLS:
        if c in off_df.columns:
            off_df[c] = pd.to_numeric(off_df[c], errors="coerce").fillna(0.0)

    def_df = df_def[season].copy()
    def_df = def_df.rename(columns={"TEAM": "Opp"})
    def_df = def_df[["Opp"] + DEF_FEATURE_COLUMNS]

    merged = off_df.merge(def_df, on="Opp", how="left")
    merged["Season"] = season

    merged_seasons.append(merged)

# remove cases where player did not play,
# they were injured, or was not active
full_df = pd.concat(merged_seasons, ignore_index=True)
full_df = full_df[full_df["MP"].notna()]
full_df = full_df[full_df["MP"] != "Inactive"]
full_df = full_df[full_df["MP"] != "Did Not Play"]

https://www.basketball-reference.com/players/h/herroty01/gamelog/2026
https://www.basketball-reference.com/players/h/herroty01/gamelog/2025
https://www.basketball-reference.com/players/h/herroty01/gamelog/2024


In [ ]:
# label all data found into one df X
# remove all cases where they played
# less than 10 minutes to remove noise
X = pd.DataFrame(full_df[FEATURE_COLUMNS + DEF_FEATURE_COLUMNS])
X = X[X["MP"] >= 10]

# make y points scored
y = pd.DataFrame(full_df['PTS'])
y.apply(pd.to_numeric, errors="coerce")
y = y.loc[X.index]

X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(0.0)

y = y.apply(pd.to_numeric, errors="coerce")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
display(X)
display(y)

,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,TOV,oPPG,dEFF,pDIFF,eDIFF,PACE,eWIN%
1,34.450000,7,17,0.412,0,6,0.000,7,11,0.636,0.412,6,7,0.857,0,7,7,3,1,113.0,111.6,-2.0,-2.0,101.3,0.398
4,33.116667,8,17,0.471,2,7,0.286,6,10,0.600,0.529,2,2,1.000,1,3,4,0,2,120.0,118.7,0.6,0.7,99.1,0.519
5,29.333333,8,15,0.533,4,6,0.667,4,9,0.444,0.667,2,3,0.667,0,3,3,2,0,108.8,115.9,-1.8,-2.1,94.1,0.448
6,32.283333,6,17,0.353,6,11,0.545,0,6,0.000,0.529,6,6,1.000,0,4,4,2,2,112.2,113.5,9.6,9.6,98.9,0.809
7,32.416667,9,15,0.600,3,5,0.600,6,10,0.600,0.700,8,8,1.000,2,3,5,7,2,115.4,118.5,-10.6,-10.8,97.4,0.321
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,35.716667,12,23,0.522,6,10,0.600,6,13,0.462,0.652,0,1,0.000,0,3,3,5,3,113.3,116.1,-2.9,-2.9,96.9,0.428
178,42.466667,12,21,0.571,2,5,0.400,10,16,0.625,0.619,9,10,0.900,1,7,8,3,3,116.4,115.8,2.6,2.6,99.9,0.564
179,39.650000,8,23,0.348,4,10,0.400,4,13,0.308,0.435,2,2,1.000,0,5,5,4,0,106.5,109.0,6.5,6.6,97.1,0.684
180,40.300000,10,20,0.500,5,12,0.417,5,8,0.625,0.625,3,3,1.000,1,5,6,6,3,109.2,111.6,11.4,11.6,97.1,0.768


,PTS
1,20
4,20
5,22
6,24
7,29
...,...
177,30
178,35
179,22
180,28


In [ ]:
def standardize(X_train, X_test):
    # Calculate the mean and standard deviation of the training set
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)

    # To prevent division by zero, set std=1 where std is zero
    std[std == 0] = 1

    # Standardize the training set
    X_train_scaled = (X_train - mean) / std

    # Use the training set's mean and std to standardize the test set
    X_test_scaled = (X_test - mean) / std

    return X_train_scaled, X_test_scaled, mean, std

In [ ]:
def gd(X, y, weights, learning_rate=0.01, n_iterations=1000, alpha=0.1):
  # apply gradient descent to train the dataset
    m, n = X.shape
    theta = np.random.randn(n, 1)

    y = y.reshape(-1, 1)
    weights = weights.reshape(-1, 1)

    mse_history = []

    for iteration in range(n_iterations):
        # Apply sample weights
        residuals = X @ theta - y
        weighted_residuals = weights * residuals

        # Full-batch weighted gradient
        gradients = (2 / m) * X.T @ weighted_residuals

        # Ridge regularization (DO NOT penalize intercept)
        ridge = 2 * alpha * theta
        ridge[0] = 0
        gradients += ridge

        # Update
        theta -= learning_rate * gradients

        # Track MSE (unweighted, for interpretability)
        mse = np.mean((X @ theta - y) ** 2)
        mse_history.append(mse)

        if iteration % 100 == 0 and iteration != 0:
            print(f"Iteration {iteration}: MSE = {mse:.4f}")

    return theta, mse_history


In [ ]:
from sklearn.model_selection import KFold

# prep into numpy for kfold
X_np = X.values.astype(float)
y_np = y.values.astype(float).reshape(-1, 1)
weights_full = make_weights(len(X_np))

kf = KFold(n_splits=5, shuffle=False)

train_losses = []
val_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np)):
    X_train, X_val = X_np[train_idx], X_np[val_idx]
    y_train, y_val = y_np[train_idx], y_np[val_idx]
    w_train = weights_full[train_idx]

    # Standardize
    X_train_s, X_val_s, mean, std = standardize(X_train, X_val)

    # Add intercept
    X_train_i = np.column_stack([np.ones(len(X_train_s)), X_train_s])
    X_val_i   = np.column_stack([np.ones(len(X_val_s)), X_val_s])

    theta, _ = gd(
        X_train_i,
        y_train,
        w_train,
        learning_rate=0.01,
        n_iterations=500,
        alpha=0.3
    )

    train_mse = np.mean((X_train_i @ theta - y_train) ** 2)
    val_mse   = np.mean((X_val_i @ theta - y_val) ** 2)

    train_losses.append(train_mse)
    val_losses.append(val_mse)

    print(f"Fold {fold+1}: Train MSE = {train_mse:.2f}, Val MSE = {val_mse:.2f}")

print("Avg Train MSE:", np.mean(train_losses))
print("Avg Val MSE:", np.mean(val_losses))

Iteration 100: MSE = 6.2090
Iteration 200: MSE = 0.7384
Iteration 300: MSE = 0.6561
Iteration 400: MSE = 0.6602
Fold 1: Train MSE = 0.66, Val MSE = 1.87
Iteration 100: MSE = 8.2906
Iteration 200: MSE = 1.0364
Iteration 300: MSE = 0.9162
Iteration 400: MSE = 0.9187
Fold 2: Train MSE = 0.92, Val MSE = 1.28
Iteration 100: MSE = 12.4946
Iteration 200: MSE = 1.3131
Iteration 300: MSE = 0.9840
Iteration 400: MSE = 0.9533
Fold 3: Train MSE = 0.94, Val MSE = 1.83
Iteration 100: MSE = 13.7683
Iteration 200: MSE = 1.3310
Iteration 300: MSE = 1.0007
Iteration 400: MSE = 0.9946
Fold 4: Train MSE = 1.00, Val MSE = 0.63
Iteration 100: MSE = 14.9000
Iteration 200: MSE = 1.6336
Iteration 300: MSE = 1.1156
Iteration 400: MSE = 1.0771
Fold 5: Train MSE = 1.07, Val MSE = 0.67
Avg Train MSE: 0.9200792100677155
Avg Val MSE: 1.2564848998575684


In [ ]:
# Final standardization
mean = X_np.mean(axis=0)
std = X_np.std(axis=0)
std[std == 0] = 1

X_scaled = (X_np - mean) / std
X_intercept = np.column_stack([np.ones(len(X_scaled)), X_scaled])

# set weight
weights_final = make_weights(len(X_np))

# and train for final feature strength
theta_final, history = gd(
    X_intercept,
    y_np,
    weights_final,
    learning_rate=0.01,
    n_iterations=600,
    alpha=0.3
)

Iteration 100: MSE = 11.0357
Iteration 200: MSE = 1.3333
Iteration 300: MSE = 1.0209
Iteration 400: MSE = 0.9744
Iteration 500: MSE = 0.9564


In [ ]:
avg = X.mean()

# get the opposing team's stats for input
opp_def = df_def[current_season].rename(columns={"TEAM": "Opp"}).set_index("Opp")

# filter by used def features
for col in DEF_FEATURE_COLUMNS:
    avg[col] = opp_def.loc[current_opponent, col]

# scale everything
player_avg_scaled = (avg.values - mean) / std
player_avg_intercept = np.insert(player_avg_scaled, 0, 1)

# apply theta to player for final prediction
prediction = player_avg_intercept @ theta_final
val_rmse = np.sqrt(np.mean(val_losses))
pred_pts = prediction.item()
lower = pred_pts - val_rmse
upper = pred_pts + val_rmse

print("Predicted points:", round(pred_pts, 2))
print("\nConclusion:")
print(f"- Expected points: {int(round(pred_pts))}")
print(f"- Typical error: ±{round(val_rmse, 2)} points")
print(f"- Likely range: [{int(lower)}, {int(upper)}]")

Predicted points: 22.64

Conclusion:
- Expected points: 23
- Typical error: ±1.12 points
- Likely range: [21, 23]
